# Schema-linking analysis

How well does the n-gram / value linker connect question phrases to the tables, columns and values that the *gold* SQL uses? (Model-independent.)

In [1]:
%matplotlib inline
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
pd.set_option("display.max_colwidth", 200)

In [2]:
from ratsql.analysis.linking_analysis import analyse_linking, gold_elements
from ratsql.schema.schema import load_schemas
from ratsql.schema.linking import SchemaLinks, describe_links
from ratsql.utils.io import read_jsonl
schemas = load_schemas(next((ROOT / "data/raw").rglob("tables.json")))
dev = read_jsonl(ROOT / "data/processed/spider/dev.jsonl")
res = analyse_linking(dev)
display(pd.Series({k: v for k, v in res.items() if not isinstance(v, dict)}).round(3))
pd.DataFrame(res["per_link_type"]).T.round(3)

examples                    1034.000
column_recall                  0.931
column_recall_exact            0.542
column_recall_value            0.243
column_precision               0.258
table_recall                   0.802
table_precision                0.711
gold_literal_coverage          0.922
mean_db_value_candidates       1.564
dtype: float64

,linked_pairs,precision
EM,1726.0,0.635
VEM,884.0,0.299
PM,4409.0,0.116
VPM,266.0,0.045
NUM,1077.0,0.019


Precision per link type = fraction of (question-token, column) links of that type whose column is used by the gold query. Exact name matches are the most reliable; partial matches are frequent but noisy — which is why RAT-SQL encodes them as *relations* the model can weigh rather than hard decisions.

In [3]:
def show(r):
    schema = schemas[r["db_id"]]
    gcols, gtabs = gold_elements(r)
    print("Q:", r["question"]); print("SQL:", r["query"])
    for line in describe_links(r["question_tokens"], schema, SchemaLinks.from_json(r["links"])):
        print("   ", line)
    print("   gold tables:", sorted(schema.tables[t].orig_name for t in gtabs))
    print("   gold columns:", sorted(schema.qualified_name(c) for c in gcols))
    print()

# successes: every gold column is linked
succ = [r for r in dev if (lambda g: g[0] and all(any(c == cc for _, cc, t in r["links"]["q_col"] if t != "NUM") for c in g[0]))(gold_elements(r))]
print(len(succ), "examples where every gold column is linked"); [show(r) for r in succ[:3]];

866 examples where every gold column is linked
Q: Show name, country, age for all singers ordered by age from the oldest to the youngest.
SQL: SELECT name ,  country ,  age FROM singer ORDER BY age DESC
    'singers' -> TABLE singer [EM]
    'singers' -> TABLE singer_in_concert [PM]
    'name' -> COLUMN stadium.Name [EM]
    'name' -> COLUMN singer.Name [EM]
    'name' -> COLUMN singer.Song_Name [PM]
    'name' -> COLUMN concert.concert_Name [PM]
    'country' -> COLUMN singer.Country [EM]
    'age' -> COLUMN singer.Age [EM]
    'singers' -> COLUMN singer.Singer_ID [PM]
    'singers' -> COLUMN singer_in_concert.Singer_ID [PM]
    'age' -> COLUMN singer.Age [EM]
   gold tables: ['singer']
   gold columns: ['singer.Age', 'singer.Country', 'singer.Name']

Q: What are the names, countries, and ages for every singer in descending order of age?
SQL: SELECT name ,  country ,  age FROM singer ORDER BY age DESC
    'singer' -> TABLE singer [EM]
    'singer' -> TABLE singer_in_concert [PM]
    '

In [4]:
# failures: gold columns with no link at all (require inference, synonyms or world knowledge)
fail = [r for r in dev if (lambda g: any(not any(c == cc for _, cc, t in r["links"]["q_col"] if t != "NUM") for c in g[0]))(gold_elements(r))]
print(len(fail), "examples with at least one unlinked gold column"); [show(r) for r in fail[:4]];

126 examples with at least one unlinked gold column
Q: What is the average, minimum, and maximum age for all French singers?
SQL: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'
    'singers' -> TABLE singer [EM]
    'singers' -> TABLE singer_in_concert [PM]
    'average' -> COLUMN stadium.Average [EM]
    'age' -> COLUMN singer.Age [EM]
    'singers' -> COLUMN singer.Singer_ID [PM]
    'singers' -> COLUMN singer_in_concert.Singer_ID [PM]
   gold tables: ['singer']
   gold columns: ['singer.Age', 'singer.Country']

Q: Show the name and the release year of the song by the youngest singer.
SQL: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1
    'singer' -> TABLE singer [EM]
    'singer' -> TABLE singer_in_concert [PM]
    'name' -> COLUMN stadium.Name [EM]
    'name' -> COLUMN singer.Name [EM]
    'name' -> COLUMN singer.Song_Name [PM]
    'name' -> COLUMN concert.concert_Name [PM]
    'release year' -> COLUMN singer.Song_release_yea

In [5]:
# value linking: gold literals recovered from the database
vals = [(r["question"], r["value_stats"]["unresolved_values"]) for r in dev if r["value_stats"]["unresolved"]]
print(f"{len(vals)} dev questions have gold literals that are not recoverable as candidates, e.g.:")
for q, v in vals[:10]: print("  ", v, "<-", q)

46 dev questions have gold literals that are not recoverable as candidates, e.g.:
   ['"France"'] <- What is the average, minimum, and maximum age for all French singers?
   ['"F"'] <- How many dog pets are raised by female students?
   ['"usa"'] <- What is the count of the car models produced in the United States?
   ['"europe"'] <- What are the names of all European countries with at least 3 manufacturers?
   ['"volvo"'] <- What is the average edispl for all volvos?
   ['1.0'] <- What are the template ids of any templates used in more than a single document?
   ['1.0'] <- find the id, name and age for visitors who visited some museums more than once.
   ['"L"'] <- What are the full names of all left handed players, in order of birth date?
   ['"L"'] <- How many different winners both participated in the WTA Championships and were left handed?
   ['"L"'] <- Find the number of left handed winners who participated in the WTA Championships.
